In [0]:
# connect to data lake
%run ../includes/storage_config.ipynb

In [0]:
df = spark.table(raw_table_name)

In [0]:
print(f'A total of {df.count()} records')
print(f'Columns: {df.columns}')

In [0]:
from pyspark.sql.functions import sum, when, col

# null counts
null_counts = df.select(
  [sum(when(col(c).isNull(),1).otherwise(0)).alias(c)for c in df.columns]  
)

display(null_counts)
# stop arrival time is empty means it's the starting point of the train, so it's not null
# stop departure time is empty means it's the end point of the train,

In [0]:
# distributions of delay, service and stops

display(df.groupBy("service_maximum_delay").count().orderBy("count"))
total_delays = df.filter(df.service_maximum_delay > 0).count()
print(f'{total_delays / df.count() * 100:.2f} % of the services are delayed')

display(df.groupBy("service_type").count().orderBy("count", ascending=False))

display(df.groupBy("stop_station_name").count().orderBy("count", ascending=False).limit(20))


In [0]:
# check duplicates
duplicate_stop_id = df.groupBy("stop_rdt_id").count().filter("count > 1").count()

duplicate_service_and_stop = df.groupBy("service_rdt_id", "stop_station_code").count() \
    .filter("count > 1").count()

total_rows = df.count()
unique_rows = df.dropDuplicates().count()
print(f"duplicate rows: {total_rows - unique_rows}")

display(duplicate_stop_id)
display(duplicate_service_and_stop)

Conclusion:
1. There's ~2 million records in one month.
2. The data is fairly complete and there are no duplicates
3. All services and trains are marked with unique identifiers
4. Fun fact, though people complain about delays a lot, 94% of the train services are on time.